# RQ6 — Class Imbalance Handling Strategies

**Research question:** How do class imbalance mitigation strategies (SMOTE, undersampling, class weighting) affect the precision-recall trade-off given the Spotify Tracks dataset's strong skew toward non-popular tracks (~89% non-popular vs ~11% popular)?

This notebook applies four strategies and compares Accuracy, Precision, Recall, F1, and ROC-AUC.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print('imbalanced-learn not available; SMOTE row will be skipped.')
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#1DB954','accent':'#D85A30','secondary':'#185FA5',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'track' in csv.lower() or 'spotify' in csv.lower(): return csv
    for c in ['tracks.csv','../tracks.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Could not find tracks.csv.')

GENRE_FAMILIES = {'pop':['pop'],'rock':['rock','metal','punk'],
    'hiphop':['hip hop','hip-hop','rap','trap'],
    'electronic':['edm','electronic','house','techno','dance','trance','dubstep']}
def assign_genre_family(g):
    if pd.isna(g) or not g: return 'other'
    s = str(g).lower()
    for fam,kws in GENRE_FAMILIES.items():
        if any(kw in s for kw in kws): return fam
    return 'other'

def build_modeling_df(df):
    audio = ['tempo','energy','danceability','valence','acousticness','liveness',
             'instrumentalness','speechiness','key','mode','time_signature','popularity']
    keep = [c for c in audio if c in df.columns]
    m = df.dropna(subset=keep).copy()
    m['popular'] = (m['popularity']>=50).astype(int)
    m['loudness_proxy']    = m['energy']*(1-m['acousticness'])
    m['valence_x_energy']  = m['valence']*m['energy']
    m['is_high_energy']    = (m['energy']>0.7).astype(int)
    m['is_danceable']      = (m['danceability']>0.7).astype(int)
    m['is_acoustic']       = (m['acousticness']>0.5).astype(int)
    m['is_instrumental']   = (m['instrumentalness']>0.5).astype(int)
    if 'genres' in m.columns:
        m['genre_family'] = m['genres'].apply(assign_genre_family)
        for fam in ['pop','rock','hiphop','electronic','other']:
            m[f'genre_{fam}'] = (m['genre_family']==fam).astype(int)
    feature_cols = [c for c in [
        'tempo','energy','danceability','valence','acousticness','liveness',
        'instrumentalness','speechiness','key','mode','time_signature',
        'loudness_proxy','valence_x_energy','is_high_energy','is_danceable',
        'is_acoustic','is_instrumental','genre_pop','genre_rock','genre_hiphop',
        'genre_electronic','genre_other'] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
if len(mdf) > 100000:
    mdf = mdf.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Modeling subset: {len(mdf):,} tracks; popular={mdf["popular"].mean():.3f}')

## 3. Analysis for RQ6

In [ ]:
X = mdf[FEATURES].fillna(0).values
y = mdf['popular'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def make_model(spw=None):
    if HAS_XGB:
        return XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
            random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False,
            n_jobs=-1, scale_pos_weight=spw if spw else 1)
    return GradientBoostingClassifier(random_state=RANDOM_STATE)

def eval_model(mdl, Xtr, ytr):
    mdl.fit(Xtr, ytr)
    yp = mdl.predict(X_test); yprob = mdl.predict_proba(X_test)[:,1]
    return {'Accuracy':round(accuracy_score(y_test,yp),3),
        'Precision':round(precision_score(y_test,yp,zero_division=0),3),
        'Recall':round(recall_score(y_test,yp,zero_division=0),3),
        'F1_Score':round(f1_score(y_test,yp,zero_division=0),3),
        'ROC_AUC':round(roc_auc_score(y_test,yprob),3)}

rows = []
row = {'Strategy':'Baseline (no resampling)'}; row.update(eval_model(make_model(), X_train, y_train)); rows.append(row)
print(f"Baseline:         F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

if HAS_SMOTE:
    sm = SMOTE(random_state=RANDOM_STATE)
    X_sm, y_sm = sm.fit_resample(X_train, y_train)
    row = {'Strategy':'SMOTE Oversampling'}; row.update(eval_model(make_model(), X_sm, y_sm)); rows.append(row)
    print(f"SMOTE:            F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

pos_idx = np.where(y_train==1)[0]; neg_idx = np.where(y_train==0)[0]
min_n = min(len(pos_idx), len(neg_idx))
under_idx = np.concatenate([np.random.choice(pos_idx, min_n, replace=False),
                             np.random.choice(neg_idx, min_n, replace=False)])
np.random.shuffle(under_idx)
row = {'Strategy':'Random Undersampling'}
row.update(eval_model(make_model(), X_train[under_idx], y_train[under_idx])); rows.append(row)
print(f"Undersampling:    F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

spw = (y_train==0).sum() / max((y_train==1).sum(), 1)
row = {'Strategy':'Class Weights (balanced)'}; row.update(eval_model(make_model(spw=spw), X_train, y_train)); rows.append(row)
print(f"Class weights:    F1={row['F1_Score']:.3f}  AUC={row['ROC_AUC']:.3f}")

imbalance_df = pd.DataFrame(rows)
imbalance_df.to_csv('table_rq6_imbalance_strategies.csv', index=False)
print('\nSaved table_rq6_imbalance_strategies.csv')
imbalance_df

## 4. Generate publication figure

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
metrics = ['Accuracy','Precision','Recall','F1_Score','ROC_AUC']
metric_colors = [COLORS['primary'],COLORS['accent'],COLORS['secondary'],COLORS['amber'],COLORS['purple']]
strategies = imbalance_df['Strategy'].tolist()
x = np.arange(len(strategies))
n = len(metrics); w = 0.14
offsets = np.linspace(-(n-1)*w/2, (n-1)*w/2, n)
for metric, color, offset in zip(metrics, metric_colors, offsets):
    ax.bar(x+offset, imbalance_df[metric], w, label=metric, color=color, edgecolor='white', linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(strategies, rotation=12, ha='right', fontsize=9)
ax.set_ylim(0, 1.0); ax.set_ylabel('Score')
ax.legend(ncol=5, loc='upper right', fontsize=8)
ax.grid(axis='y', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)
fig.suptitle('Figure 6.1 — Class Imbalance Handling Strategies (Spotify Tracks)',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq6_imbalance_strategies.pdf')
plt.savefig('fig_rq6_imbalance_strategies.png')
plt.show()
print('Saved fig_rq6_imbalance_strategies.pdf / .png')

## 5. Conclusion

Given the strong class imbalance in the Spotify Tracks dataset (~89% non-popular), the baseline achieves high Accuracy by predicting mostly non-popular but suffers low Recall on the minority class. SMOTE and class weights significantly improve Recall and F1 on popular tracks at a modest cost in Accuracy and Precision. For applications where catching popular tracks matters (e.g. recommendation), SMOTE or class weights are clearly preferable over the baseline.